# A2 · Decisión ZAP (cielo)

**Spec:** [`docs/spec_A2_codex_sky_zap.md`](../docs/spec_A2_codex_sky_zap.md)  |  **Bloque:** A · Reducción  |  **Run de este set:** `ROXs42Bb_realigned`

Decide y aplica (o descarta) la sustracción de cielo con ZAP.

| | |
|---|---|
| **Entrada** | Cubo reducido |
| **Salida (QC/productos)** | Cubo con cielo tratado (sin QC separado en este run) |
| **Consume aguas abajo** | A3, A4 |


## Qué es ZAP y por qué se necesita

**ZAP** (*Zurich Atmosphere Purge*, [Soto, Lilly, Bacon, Richard & Conseil 2016, MNRAS 458, 3210](https://doi.org/10.1093/mnras/stw474)) es una sustracción de **residuos de cielo** para MUSE basada en PCA. El pipeline (`muse_scipost --skymethod=model`, [Weilbacher et al. 2020, A&A 641, A28](https://doi.org/10.1051/0004-6361/202037855) §3.2) ya resta un modelo de cielo — la 3ª figura de abajo lo verifica en el log de esorex —, pero el **airglow** (líneas de OH y [O I] atmosféricas) es intenso y **varía en el tiempo** entre la exposición de ciencia y el modelo → suelen quedar **residuos de skylines**. ZAP construye una base PCA con los spaxels de **cielo** (con las fuentes enmascaradas) y elimina las componentes que describen esos residuos, dejando la señal astrofísica.

**Por qué importa aquí:** un residuo de skyline mal restado puede **imitar o contaminar** una línea espectral. Buscamos una línea débil de Hα en el compañero, así que el cielo residual es un contaminante de primer orden.

**El peligro (por qué NO se aplica a ciegas):** ZAP necesita suficientes spaxels de cielo *reales*. En el **campo diminuto de NFM**, con una estrella brillante y su compañero, la fracción de cielo es baja y las eigencomponentes pueden **absorber señal del compañero** — incluso *fabricar o borrar* una línea en Hα. Regla del proyecto: ante la duda, **no tocar la señal**.

**Decisión pre-registrada** (`musepipe.reduction.sky_zap.classify_zap_decision`), por métrica, no por juicio. `R` = RMS mediano en ventanas de skyline ÷ RMS mediano en continuo, medido en aperturas de cielo vacías:

| Condición | Decisión |
|---|---|
| `R ≤ 1.5` | **no necesario** → `zap_applied = False` |
| `R > 2.0` | **necesario** → `zap_applied = True` |
| `1.5 < R ≤ 2.0` | zona gris → checkpoint (no aplicar, preguntar) |
| fracción de cielo `< 0.25` | cielo insuficiente → checkpoint (no aplicar) |

Los parámetros de ZAP quedan en *default*; no se itera buscando 'el mejor resultado'.


## Cómo ejecutar de forma independiente

Etapa de **reducción**: la celda de abajo resuelve el comando real para **este objeto** a partir de su `chain.reduction_profile` y de su config, y puede lanzarlo. Son trabajos largos (ver coste), así que se lanzan en segundo plano con el log a la vista; el notebook no se bloquea.

Si algún dato no está declarado en el config del run, la celda lo dice y **no lanza** en vez de inventarse una ruta.

Comando histórico de referencia:

```bash
conda activate MUSE
bash scripts/sky_zap.sh
```


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
cmds, target_run, missing = nb.launch_command('A2', RUN_ID)
print('run que ejecuta esta etapa:', target_run)
# Una etapa puede necesitar VARIOS comandos: A4 son cuatro (M1/M2, M3,
# M4/M5 y el recalculo de lo derivado). Se lanzan EN ORDEN y se abortan
# al primer fallo; publicar solo uno es lo que dejo a un objeto un mes
# sin M4 ni M5.
print(f'comandos resueltos para este objeto ({len(cmds)}):')
for i, c in enumerate(cmds, 1):
    print(f'   {i}. {c}')
if not cmds:
    print('   (sin plantilla)')
if missing:
    print()
    print('NO se puede lanzar: faltan datos en el config del run.')
    print('   sin resolver:', ', '.join(missing))
    print(f'   declara esas claves en runs/{target_run}/config/config.json')

RUN = False   # -> True para LANZAR (trabajo largo: revisa el coste arriba)

if RUN and cmds and not missing:
    import subprocess, time
    from pathlib import Path
    log = Path(nb.run_dir(target_run)) / 'logs' / f'a2_launch.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    # `&&` da la secuencia y el aborto al primer fallo sin cambiar lo que
    # esta celda ya hacia: un lanzamiento de fondo, un pid, un log. El log
    # se abre en modo append porque ahora escriben varios pasos.
    with open(log, 'a') as fh:
        proc = subprocess.Popen(' && '.join(cmds), shell=True,
                                cwd=str(nb.project_root()),
                                stdout=fh, stderr=subprocess.STDOUT)
    print(f'lanzados {len(cmds)} comandos en segundo plano (pid {proc.pid}); log -> {log}')
    print('sigue el progreso con:  !tail -f', log)
elif RUN:
    print('RUN=True pero no hay comandos o hay datos sin resolver: no se lanza nada.')
else:
    print()
    print('Modo auditoría (RUN=False): abajo se carga el QC existente.')


## Resultados que llevaron a la conclusión

Métrica **M4 de cielo** del QC del cubo (`stages/stage00q_qc.json`) aplicada a la regla de decisión pre-registrada.


In [ ]:
qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
m4 = qc.get('m4_sky', {})
R = m4.get('R')
sky_frac = m4.get('sky_fraction')   # QCs antiguos no lo persisten (gap documentado)
print(f'm4_sky.R = {R}   (RMS skyline / RMS continuo en aperturas vacías)   [status {m4.get("status")}]')

# Decisión con la implementación OFICIAL (no reimplementada aquí):
try:
    from musepipe.reduction.sky_zap import classify_zap_decision
    if R is None:
        print('sin dato de R en el QC -> decisión no evaluable')
    else:
        if sky_frac is None:
            print('AVISO: el QC no persiste sky_fraction -> la rama '
                  'insufficient_sky (<0.25) no es re-derivable aquí; se evalúa solo la rama R.')
        d = classify_zap_decision(float(R), float(sky_frac) if sky_frac is not None else 1.0)
        print(f'classify_zap_decision: {d.decision}  ->  zap_applied = {d.zap_applied}'
              f'  (checkpoint_required={d.checkpoint_required})')
except Exception as e:
    print('No se pudo importar musepipe (kernel sin la pila científica):', type(e).__name__, e)
    print('Regla pre-registrada (espejo de classify_zap_decision): R<=1.5 no necesario | '
          'R>2.0 necesario | zona gris -> checkpoint | sky_fraction<0.25 -> checkpoint')

print()
print('Corroboración (nota del QC):')
print('  ', qc.get('note'))
print('M1/M2 se midieron del SKY_SPECTRUM cacheado (airglow, 32 exp,',
      qc.get('m1_wavelength', {}).get('n_measurements'), 'medidas) porque el')
print('cubo restado de cielo tiene <8 skylines usables.')


## De dónde sale R: los datos, la zona y de quién es la prueba

**Un solo FITS:** el cubo de A1 (`cube.file` del QC), extensión **DATA** (la STAT no
interviene en R). R se mide sobre los **spaxels de cielo** de ese cubo — no hay varios
archivos. *(Los `SKY_SPECTRUM` cacheados son de M1/M2, no de R.)*

### Procedencia de la prueba (esto es lo que cita la figura)

| Pieza | Origen |
|---|---|
| **ZAP**, el método que se decide aplicar o no | [Soto, Lilly, Bacon, Richard & Conseil 2016, MNRAS 458, 3210](https://doi.org/10.1093/mnras/stw474) |
| **La resta de cielo que ya hizo el DRS** (lo que R audita) | [Weilbacher et al. 2020, A&A 641, A28](https://doi.org/10.1051/0004-6361/202037855) §3.2; [Streicher et al. 2011, ASPC 442, 257](https://ui.adsabs.harvard.edu/abs/2011ASPC..442..257S) |
| **Qué λ son skyline** (5577.3, 6300.3, 6363.8 Å, banda O₂ 6864–6960, bandas OH 7240–9300) | atlas de airglow del VLT, [Hanuschik 2003, A&A 407, 1157](https://doi.org/10.1051/0004-6361:20030783) |
| **El cociente `R` y los umbrales 1.5 / 2.0** | **métrica de este proyecto**, `docs/spec_A2_codex_sky_zap.md` §3 — **no** viene de un paper |

Esa última fila importa: `R` es una regla **pre-registrada nuestra**, no un test estándar de
la literatura. Lo que la respalda es que se fijó antes de mirar el dato y que la conclusión
sobrevive a una versión de la prueba libre del sesgo que se explica abajo.

### El sesgo del `R` global, y la prueba que no lo tiene

`R` compara la mediana del RMS en ventanas de skyline contra la mediana en dos ventanas de
continuo (5100–5500, 6600–6800 Å). Pero el RMS por canal **no es plano en λ**: la curva de
respuesta de MUSE hunde la transmisión en el azul, así que el mismo residuo en cuentas sale
amplificado allí. Medido en este cubo: RMS ≈ 3.0 en 5100–5500 Å frente a ≈ 1.6 en
6600–6800 Å. Como casi todas las ventanas de skyline caen en el rojo (donde la respuesta es
plana), **parte de que `R < 1` es la curva de respuesta, no cielo limpio**.

Por eso el panel (b) repite la prueba **dentro de cada banda**: `R_local` = RMS de los
canales que caen sobre airglow **brillante** ÷ RMS de los canales vecinos de airglow
**débil**, en la misma banda de 575 Å (ranking de brillo tomado del `SKY_SPECTRUM` del DRS,
que es el airglow real antes de restarlo; si no está en disco, del propio residuo del cubo).
Ambos numerador y denominador comparten la misma respuesta → el cociente es limpio. Si los
residuos de skyline fueran un problema, `R_local` se despegaría de 1 en las bandas de OH.

> Necesita el kernel **MUSE** (astropy) y el cubo en disco. Usa la **máscara oficial** de A2
> (`zap_source_mask.fits`, el mismo FITS que escribió la etapa), así que el `R` reproducido
> aquí coincide con el del QC salvo redondeo; la celda imprime de dónde salió la máscara.


In [ ]:
try:
    MAKE_PLOT = True   # carga el cubo (~590 MB en RAM) via astropy; requiere kernel MUSE

    # Procedencia de la prueba, al pie de cada figura de A2. La ultima linea es la
    # honesta: R es una regla de este proyecto, no un test publicado.
    A2_REFS = (
        "ZAP: Soto et al. 2016, MNRAS 458, 3210  ·  "
        "resta de cielo del DRS: Weilbacher et al. 2020, A&A 641, A28 (3.2), Streicher et al. 2011, ASPC 442, 257  ·  "
        "lineas/bandas de airglow: Hanuschik 2003, A&A 407, 1157  ·  "
        "el cociente R y los umbrales 1.5/2.0: metrica de este proyecto (docs/spec_A2_codex_sky_zap.md sec.3), sin paper"
    )


    def _a2_load():
        """Cubo + mascara OFICIAL de cielo de A2, una sola vez, reusados por las 3 figuras."""
        global _A2
        if "_A2" in globals() and _A2.get("data") is not None:
            return _A2
        import numpy as np
        from pathlib import Path
        from astropy.io import fits

        qc_q = nb.load_qc("stages/stage00q_qc.json", RUN_ID)
        cube_path = qc_q.get("input_cube") or qc_q.get("cube", {}).get("file")
        if not cube_path:
            raise RuntimeError("el QC de A4 no declara el cubo de entrada")
        # El QC guarda unas rutas absolutas y otras RELATIVAS a la raiz del repo
        # (ROXs 12 b: /mnt/2TB/...; ROXs 42B b: runs/<run>/cube_telcorr.fits). El cwd
        # del notebook es notebooks/<objeto>/ -- la celda de setup anade la raiz al
        # sys.path pero NO hace chdir --, asi que una relativa no resuelve sola.
        cube_path = Path(cube_path)
        if not cube_path.is_absolute():
            cube_path = nb.project_root() / cube_path
        print("FITS usado:", cube_path)
        if not cube_path.exists():
            raise FileNotFoundError(
                f"el cubo que declara el QC no esta en disco: {cube_path}"
                "  (revisa cube.file del QC de A4 para este run)")

        hdul = fits.open(cube_path, memmap=True)
        data = np.asarray(hdul[1].data, dtype=np.float32)
        hd = hdul[1].header
        n3 = hd["NAXIS3"]
        wave = hd["CRVAL3"] + (np.arange(n3) - (hd["CRPIX3"] - 1)) * hd["CD3_3"]
        wl = np.nanmedian(data, axis=0)
        finite = np.isfinite(wl)

        # Mascara de cielo: la OFICIAL que escribio la etapa, no una aproximacion.
        # Orden explicito y anunciado; si no hay ninguna se reconstruye con la funcion
        # canonica y se dice. Nunca un percentil silencioso.
        mask_path, origin = None, None
        for qc_name, key in (("stages/stage00s_qc.json", ("mask", "file")),
                             ("stages/stage00q_qc.json", ("m4_sky", "source_mask"))):
            try:
                _qc = nb.load_qc(qc_name, RUN_ID)
            except FileNotFoundError:
                continue
            cand = (_qc.get(key[0]) or {}).get(key[1])
            if not cand:
                continue
            cand = Path(cand)
            if not cand.is_absolute():          # relativa a la raiz, igual que el cubo
                cand = nb.project_root() / cand
            if cand.exists():
                mask_path, origin = cand, f"{qc_name}:{key[0]}.{key[1]}"
                break
        if mask_path is not None:
            source = np.asarray(fits.getdata(mask_path)) > 0
            print(f"mascara de cielo: FITS oficial de la etapa ({origin})")
        else:
            from musepipe.reduction.sky_zap import build_source_mask, SourceRegion
            _sy, _sx = np.unravel_index(np.nanargmax(wl), wl.shape)
            _regs = [SourceRegion(name="primary", yx=(float(_sy), float(_sx)),
                                  radius_px=20.0, auto_halo=True)]
            try:
                _gc = nb.load_qc("stages/stage00s_qc.json", RUN_ID).get("growth_curve", {})
                _cyx = _gc.get("companion_masked_yx")
            except FileNotFoundError:
                _cyx = None
            if _cyx:
                _regs.append(SourceRegion(name="companion", yx=(float(_cyx[0]), float(_cyx[1])),
                                          radius_px=8.0, auto_halo=False))
            source, _radii = build_source_mask(wl, _regs)
            print("mascara de cielo: RECONSTRUIDA con build_source_mask (no habia FITS en el QC);",
                  f"primaria en {(_sy, _sx)}, companero {_cyx}, radios {_radii}")
        sky_mask = finite & ~source

        # Mediana del cielo por spaxel, canal a canal, por trozos (no duplica el cubo).
        idx = np.where(sky_mask.ravel())[0]
        sky_med = np.full(n3, np.nan)
        for c0 in range(0, n3, 200):
            blk = data[c0:c0 + 200].reshape(min(200, n3 - c0), -1)[:, idx]
            sky_med[c0:c0 + blk.shape[0]] = np.nanmedian(blk, axis=1)

        pix = abs(float(hd.get("CD1_1") or hd.get("CDELT1", 0.0))) * 3600.0
        sy, sx = np.unravel_index(np.nanargmax(wl), wl.shape)
        _A2 = dict(data=data, hd=hd, wave=wave, wl=wl, sky_mask=sky_mask, sky_med=sky_med,
                   n3=n3, pix=pix, star_yx=(int(sy), int(sx)), cube_path=cube_path,
                   qc_q=qc_q, hdul=hdul)
        print(f"cielo: {sky_mask.mean():.4f} del campo ({sky_mask.sum()} spaxels)   "
              f"escala {pix:.4f}\"/px   primaria en (y,x)={(int(sy), int(sx))}")
        return _A2


    if MAKE_PLOT:
        try:
            import numpy as np
            import matplotlib.pyplot as plt
            from musepipe.reduction.sky_zap import (
                compute_sky_residual_metrics, channel_rms_from_mask,
                SKYLINE_WINDOWS, CONTINUUM_WINDOWS)

            A = _a2_load()
            wave = A["wave"]
            m = compute_sky_residual_metrics(A["data"].astype(np.float64), wave, A["sky_mask"])
            R = m["R_skyline_over_continuum"]
            rms = m["channel_rms"]
            _qcR = A["qc_q"].get("m4_sky", {}).get("R")
            print(f"R reproducido = {R:.3f}   (oficial m4_sky.R = {_qcR})")

            # El sesgo de la respuesta, en numeros: las dos ventanas de continuo no son
            # equivalentes entre si, y las de skyline viven casi todas en el rojo plano.
            print("\nRMS mediano por ventana (asi se ve el sesgo de la curva de respuesta):")
            for tag, wins in (("skyline ", SKYLINE_WINDOWS), ("continuo", CONTINUUM_WINDOWS)):
                for a, b in wins:
                    s = (wave >= a) & (wave <= b) & np.isfinite(rms)
                    print(f"  {tag} {a:7.1f}-{b:7.1f} A  n={int(s.sum()):5d}  RMS={np.median(rms[s]):7.3f}")

            # R_local: misma prueba dentro de cada banda, inmune a la respuesta.
            # Ranking de brillo del airglow: SKY_SPECTRUM del DRS si esta en disco
            # (es el cielo REAL antes de restarlo); si no, el residuo del propio cubo.
            from pathlib import Path
            import glob as _glob
            _ref, _ref_src = None, None
            _root = Path(A["cube_path"]).parent.parent
            _cand = sorted(_glob.glob(str(_root / "*" / "products" / "*" / "SKY_SPECTRUM_*.fits")))
            if _cand:
                from astropy.io import fits as _fits
                _t = _fits.open(_cand[0])[1].data
                _ref = np.interp(wave, _t["lambda"], _t["data"])
                _ref_src = f"SKY_SPECTRUM del DRS ({Path(_cand[0]).parent.name})"
            else:
                _ref = A["sky_med"]
                _ref_src = "residuo de cielo del propio cubo (no hay SKY_SPECTRUM en disco)"
            print(f"\nR_local: ranking de airglow = {_ref_src}")
            edges = np.linspace(wave[0], wave[-1], 9)
            rloc, rlab = [], []
            for i in range(8):
                s = (wave >= edges[i]) & (wave < edges[i + 1]) & np.isfinite(rms) & np.isfinite(_ref)
                v, r = _ref[s], rms[s]
                hi, lo = v >= np.percentile(v, 70), v <= np.percentile(v, 30)
                rloc.append(float(np.median(r[hi]) / np.median(r[lo])))
                rlab.append(f"{edges[i]:.0f}\n{edges[i + 1]:.0f}")
                print(f"  {edges[i]:6.0f}-{edges[i + 1]:6.0f} A   R_local = {rloc[-1]:5.3f}"
                      f"   (brillante {np.median(r[hi]):6.3f} / debil {np.median(r[lo]):6.3f})")
            print(f"  -> maximo {max(rloc):.3f} sobre las 8 bandas, umbral de la etapa 1.5")

            fig, axes = plt.subplots(2, 2, figsize=(14.5, 8.6),
                                     gridspec_kw={"height_ratios": [1, 1]})
            ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

            # (a) la prueba tal como la define la etapa
            ax1.plot(wave, rms, lw=0.5, color="0.35")
            for i, (a, b) in enumerate(SKYLINE_WINDOWS):
                ax1.axvspan(a, b, color="tab:red", alpha=0.18,
                            label="ventana skyline (Hanuschik 2003)" if i == 0 else None)
            for i, (a, b) in enumerate(CONTINUUM_WINDOWS):
                ax1.axvspan(a, b, color="tab:green", alpha=0.25,
                            label="ventana continuo" if i == 0 else None)
            ax1.axhline(m["skyline_rms_median"], color="tab:red", ls="--", lw=1)
            ax1.axhline(m["continuum_rms_median"], color="tab:green", ls="--", lw=1)
            ax1.set_xlabel("lambda [A]"); ax1.set_ylabel("RMS por canal (cielo)")
            ax1.set_title(f"(a) prueba R de la etapa: med(skyline)/med(continuo) = {R:.3f}"
                          f"  [umbral 1.5]", fontsize=9.5)
            ax1.set_ylim(0, np.nanpercentile(rms, 99)); ax1.legend(fontsize=7.5)

            # (b) la misma prueba sin el sesgo de la respuesta
            _col = ["tab:green" if v <= 1.5 else "tab:red" for v in rloc]
            ax2.bar(range(8), rloc, color=_col, alpha=0.8)
            ax2.axhline(1.0, color="0.4", ls=":", lw=1, label="1 = sin exceso sobre el vecino")
            ax2.axhline(1.5, color="tab:red", ls="--", lw=1, label="umbral 1.5 de la etapa")
            ax2.set_xticks(range(8)); ax2.set_xticklabels(rlab, fontsize=6.5)
            ax2.set_ylim(0, max(1.75, max(rloc) * 1.15))
            ax2.set_xlabel("banda [A]"); ax2.set_ylabel("R_local")
            ax2.set_title("(b) misma prueba DENTRO de cada banda (airglow brillante / vecino debil):\n"
                          "sin sesgo de la curva de respuesta", fontsize=9.5)
            ax2.legend(fontsize=7.5)

            # (c) la zona de cielo realmente usada
            _ny, _nx = A["wl"].shape
            _ext = [-0.5, _nx - 0.5, -0.5, _ny - 0.5]
            ax3.imshow(np.log10(np.clip(A["wl"], 1, None)), origin="lower", cmap="gray", extent=_ext)
            ov = np.zeros((*A["wl"].shape, 4)); ov[A["sky_mask"]] = [0.1, 0.5, 1.0, 0.5]
            ax3.imshow(ov, origin="lower", extent=_ext)
            _sy, _sx = A["star_yx"]; _pix = A["pix"]
            if _pix > 0:
                for _as in (0.5, 1.0, 2.0, 3.0):
                    _r = _as / _pix
                    ax3.add_patch(plt.Circle((_sx, _sy), _r, fill=False, color="tab:orange",
                                             lw=0.8, ls="--"))
                    ax3.annotate(f"{_as:g}\"", (_sx, _sy + _r), color="tab:orange",
                                 fontsize=6.5, ha="center", va="bottom")
            ax3.plot(_sx, _sy, "+", color="tab:cyan", ms=8, mew=1.4)
            ax3.set_xlim(_ext[0], _ext[1]); ax3.set_ylim(_ext[2], _ext[3])
            ax3.set_xlabel("x [px]"); ax3.set_ylabel("y [px]"); ax3.tick_params(labelsize=7)
            ax3.set_title(f"(c) zona de cielo oficial (azul), {A['sky_mask'].mean():.3f} del campo"
                          f" · {_pix:.4f}\"/px", fontsize=9.5)

            # (d) lo que queda tras la resta del DRS
            sky_med = A["sky_med"]; _fin = np.isfinite(sky_med)
            ax4.plot(wave, sky_med, lw=0.5, color="tab:purple")
            ax4.axhline(0, color="tab:red", ls="--", lw=1.0, label="cero (cielo perfecto)")
            _mm = float(np.median(sky_med[_fin]))
            ax4.axhline(_mm, color="0.35", ls=":", lw=1.0, label=f"mediana global {_mm:+.3f}")
            _q = np.nanpercentile(sky_med[_fin], [1, 99])
            ax4.set_ylim(_q[0] - 0.2 * abs(_q[0]), _q[1] + 0.2 * abs(_q[1]))
            ax4.set_xlabel("lambda [A]"); ax4.set_ylabel("mediana del cielo por spaxel")
            ax4.legend(fontsize=7.5)
            ax4.set_title("(d) lo que queda tras la resta de cielo del DRS\n"
                          "(su origen y su forma en lambda, en la 3a figura)", fontsize=9.5)

            fig.tight_layout(rect=[0, 0.055, 1, 1])
            fig.text(0.005, 0.012, A2_REFS, fontsize=6.2, color="0.25", wrap=True)
            outdir = nb.run_dir(RUN_ID) / "plots" / "a2_m4"
            outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / "m4_R.png", dpi=110)
            print("\nfigura ->", outdir / "m4_R.png")
            plt.show()
        except Exception as e:
            print("No se pudo generar el plot:", type(e).__name__, e)
            print("Necesita el kernel MUSE (astropy) y el cubo en disco (campo cube.file del QC de A4).")
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## ¿La zona de cielo vale para todas las longitudes de onda?

**Por construcción, la etapa usa UNA sola máscara para todo el rango.** `build_source_mask`
recibe la **imagen de luz blanca** (mediana sobre los 3681 canales) y el resultado se aplica
igual en 4750 Å que en 9350 Å. Eso es una **suposición**, y hay una razón física para
dudar de ella: el halo AO es **cromático** — el Strehl sube con λ, así que en el azul una
fracción mayor de la luz de la primaria vive en el halo ancho y podría invadir la zona que
llamamos cielo (es el mismo efecto que documenta `docs/` para la sobre-sustracción de
continuo en C3).

La celda mide ese coste en vez de suponerlo: parte el rango en **8 bandas de 575 Å**, aplica
en cada una **el mismo criterio canónico** (`build_source_mask` sobre la imagen mediana de
esa banda, mismos `threshold_sigma` y `dilation_px` del QC, y el radio del halo de la
primaria **remedido** por banda con `estimate_halo_radius`), y compara la máscara por banda
con la máscara global:

- `sky_fraction` por banda — ¿queda cielo suficiente (≥ 0.25) en todas?
- `r_halo` de la primaria por banda — ¿crece el halo hacia el azul?
- **fuga** — la fracción de la zona de cielo **global** que *esta* banda llamaría fuente.
- **Jaccard** (informativo) — máscara-de-banda ∩ global ÷ unión; 1.0 = idénticas.

El veredicto se apoya en la **fuga**, no en el Jaccard, porque el riesgo es **asimétrico**:
que una banda pida *más* halo significa que la máscara única está metiendo halo dentro del
cielo (malo); que pida *menos* solo significa que se podría haber usado más cielo (inocuo). El
Jaccard mezcla los dos sentidos. La celda pide **fuga ≤ 5 %** en las 8 bandas y
`sky_fraction ≥ 0.25`, y si alguna se sale lo dice en vez de dar la máscara única por buena.

Como referencia de lo que salió al escribir esto: en **ROXs12b** `sky_fraction` 0.420–0.447,
halo 84–86 px, fuga máxima ~3 %; en **ROXs42Bb** 0.420–0.485, halo 81–86 px, fuga del mismo
orden — su Jaccard baja a 0.89 en la banda azul extrema, pero es porque allí el halo **se
encoge** (más cielo, no menos), que es el sentido inocuo.

O sea: la suposición sale barata, la zona de cielo es esencialmente la misma en todo el rango
y la máscara única no es un atajo que cambie la decisión de ZAP. Lo que **sí** depende de λ es
*cuánta luz de halo queda dentro* de esa zona (el nivel, no la geometría) — y eso es
exactamente lo que mide la figura siguiente.


In [ ]:
try:
    if MAKE_PLOT:
        try:
            import numpy as np
            import matplotlib.pyplot as plt
            from musepipe.reduction.sky_zap import build_source_mask, SourceRegion

            A = _a2_load()
            wave, data, wl = A["wave"], A["data"], A["wl"]
            sky_global = A["sky_mask"]

            # Los knobs salen del QC de la etapa, no de literales: si la etapa corrio con
            # otro umbral, esta figura tiene que seguirlo.
            _thr, _dil = 3.0, 2
            _cyx = None
            try:
                _s = nb.load_qc("stages/stage00s_qc.json", RUN_ID)
                _thr = float(_s.get("mask", {}).get("threshold_sigma", _thr))
                _dil = int(_s.get("mask", {}).get("dilation_px", _dil))
                _cyx = _s.get("growth_curve", {}).get("companion_masked_yx")
            except FileNotFoundError:
                print("[A2 no ha corrido para este objeto: se usan los defaults 3.0 sigma / 2 px]")
            print(f"criterio por banda: threshold_sigma={_thr}  dilation_px={_dil}  companero={_cyx}")

            _sy, _sx = A["star_yx"]
            regs = [SourceRegion(name="primary", yx=(float(_sy), float(_sx)),
                                 radius_px=20.0, auto_halo=True)]
            if _cyx:
                regs.append(SourceRegion(name="companion", yx=(float(_cyx[0]), float(_cyx[1])),
                                         radius_px=8.0, auto_halo=False))

            edges = np.linspace(wave[0], wave[-1], 9)
            fig, axes = plt.subplots(2, 4, figsize=(16.5, 8.4))
            _ny, _nx = wl.shape
            _ext = [-0.5, _nx - 0.5, -0.5, _ny - 0.5]
            # `fuga` es la metrica que importa, y es ASIMETRICA: la fraccion de la zona de
            # cielo global que ESTA banda llamaria fuente (halo que crece -> la mascara unica
            # mete halo en el cielo). Que la banda pida MENOS fuente es inocuo: solo significa
            # que se podria haber usado mas cielo. El Jaccard no distingue los dos sentidos.
            print("\n  banda [A]      sky_fraction  r_halo [px]  Jaccard  fuga(halo->cielo)  mediana del cielo")
            rows = []
            for i, ax in enumerate(axes.ravel()):
                sel = (wave >= edges[i]) & (wave < edges[i + 1])
                img = np.nanmedian(data[sel], axis=0)
                src, radii = build_source_mask(img, regs, threshold_sigma=_thr, dilation_px=_dil)
                sky = np.isfinite(img) & ~src
                jac = float((sky & sky_global).sum() / max(1, (sky | sky_global).sum()))
                leak = float((sky_global & ~sky).sum() / max(1, sky_global.sum()))
                med = float(np.nanmedian(img[sky]))
                rows.append((edges[i], edges[i + 1], float(sky.mean()),
                             float(radii.get("primary", np.nan)), jac, med, leak))
                print(f"  {edges[i]:6.0f}-{edges[i + 1]:6.0f}     {sky.mean():.3f}"
                      f"         {radii.get('primary', float('nan')):5.1f}"
                      f"        {jac:.3f}        {leak:.4f}            {med:+.3f}")

                ax.imshow(np.log10(np.clip(img, 1, None)), origin="lower", cmap="gray", extent=_ext)
                ov = np.zeros((*img.shape, 4)); ov[sky] = [0.1, 0.5, 1.0, 0.45]
                ax.imshow(ov, origin="lower", extent=_ext)
                # en rojo, donde la mascara de ESTA banda difiere de la global
                dif = sky ^ sky_global
                if dif.any():
                    ov2 = np.zeros((*img.shape, 4)); ov2[dif] = [1.0, 0.2, 0.2, 0.95]
                    ax.imshow(ov2, origin="lower", extent=_ext)
                if A["pix"] > 0:
                    ax.add_patch(plt.Circle((_sx, _sy), 1.0 / A["pix"], fill=False,
                                            color="tab:orange", lw=0.7, ls="--"))
                ax.plot(_sx, _sy, "+", color="tab:cyan", ms=6, mew=1.2)
                ax.set_xlim(_ext[0], _ext[1]); ax.set_ylim(_ext[2], _ext[3])
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(f"{edges[i]:.0f}-{edges[i + 1]:.0f} A\n"
                             f"cielo {sky.mean():.3f} · r_halo {radii.get('primary', float('nan')):.0f} px"
                             f" · fuga {100 * leak:.1f}%", fontsize=8.5)

            _fr = [r[2] for r in rows]; _rh = [r[3] for r in rows]
            _jj = [r[4] for r in rows]; _lk = [r[6] for r in rows]
            print(f"\n  sky_fraction {min(_fr):.3f}-{max(_fr):.3f} (minimo de la etapa 0.25)"
                  f"   r_halo {min(_rh):.0f}-{max(_rh):.0f} px"
                  f"   Jaccard minimo {min(_jj):.3f}   fuga maxima {max(_lk):.4f}")
            # El veredicto lo dicta la medida, no el texto.
            _ok_sky = min(_fr) >= 0.25
            _ok_geom = max(_lk) <= 0.05
            if _ok_geom and _ok_sky:
                print(f"  -> la geometria de la zona de cielo NO depende de lambda de forma relevante:"
                      f" en el peor caso solo el {100 * max(_lk):.1f}% del cielo global seria fuente"
                      f" para su banda (umbral 5%), y queda cielo de sobra en las 8."
                      f" La mascara unica de la etapa esta justificada.")
            else:
                _b = rows[int(np.argmax(_lk))]
                print(f"  -> ATENCION: la mascara unica NO queda justificada por la medida"
                      f" (fuga maxima {100 * max(_lk):.1f}% en {_b[0]:.0f}-{_b[1]:.0f} A"
                      f"{'' if _ok_sky else f', sky_fraction minima {min(_fr):.3f} < 0.25'}):"
                      " esa banda mete halo en el cielo y hay que revisarla antes de usar R"
                      " como decision global.")
            print("  (en rojo, los spaxels donde la mascara de la banda difiere de la global)")

            fig.suptitle("Zona de cielo (azul) banda a banda, mismo criterio canonico que la etapa"
                         " — rojo = difiere de la mascara global", fontsize=11)
            fig.tight_layout(rect=[0, 0.045, 1, 0.96])
            fig.text(0.005, 0.010, A2_REFS, fontsize=6.2, color="0.25", wrap=True)
            outdir = nb.run_dir(RUN_ID) / "plots" / "a2_m4"
            outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / "sky_zone_por_banda.png", dpi=110)
            print("figura ->", outdir / "sky_zone_por_banda.png")
            plt.show()
        except NameError:
            print("Ejecuta primero la celda anterior (define _a2_load y A2_REFS).")
        except Exception as e:
            print("No se pudo generar el plot:", type(e).__name__, e)
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## ¿Restó el cielo esorex? Sí — y por eso el suelo que queda no es cielo

**Sí, `muse_scipost` ya restó cielo**, con su método por defecto y sin que nosotros lo
toquemos (spec_A1: el único override es `--save`). La evidencia no es de memoria: está en el
log de esorex de cada exposición, y la celda lo imprime literal. Lo que hizo, en orden
([Weilbacher et al. 2020](https://doi.org/10.1051/0004-6361/202037855) §3.2):

1. `--skymethod="model"` (default). Reconstruye una imagen blanca intermedia de **esa**
   exposición y crea su propia máscara de cielo con la **fracción más oscura** del campo:
   `--skymodel_fraction=0.1` tras ignorar `--skymodel_ignore=0.05`. En este objeto el
   `SKY_MASK` que guardó cubre el **9.1 %** del campo.
2. Ajusta las líneas de airglow partiendo del `SKY_LINES` de entrada (con la LSF local) y las
   resta; `Sky line fit finished successfully. Offset -0.021 A (at 4746 A) ... -0.042 A (at 9355 A)`.
3. `No sky continuum given, create it from the data` → construye el **continuo de cielo** a
   partir del residuo de ese mismo 9.1 % y lo resta también,
   `Cutting data to 4750.000...9349.688 Angstrom` — o sea **todo el rango**, sin extremos sin
   tratar.

Y aquí está el punto fino de NFM: el campo son **5″**. El 9.1 % más oscuro que el DRS llamó
«cielo» está a solo ~2.5–3.6″ de una estrella brillante, así que **ya contiene halo AO**. El
DRS resta ese nivel a todo el campo. Nuestra zona de cielo es más ancha (43 % del campo) y
está *más cerca* de la primaria en promedio → lo que queda dentro es el **exceso de halo**
respecto de esa referencia, no cielo mal restado. De ahí el suelo **positivo** (mediana ≈ +1.6
en unidades del cubo sobre los 3681 canales; el `median_bias` = 1.14 del QC de A4 es el mismo
efecto medido sobre sus 32 aperturas vacías, que están más lejos de la primaria).

### Por qué el azul y el rojo salen distintos

Los tres paneles de abajo separan las causas, y las tres son medibles:

- **(b) La prueba que decide qué es:** perfil radial del suelo, en anillos fuera de la
  máscara. Un cielo mal restado sería **plano** con el radio; un halo **cae**. Medido: cae
  **×2.6** en el azul y **×5.5** en el rojo entre ~2.3″ y ~3.5″ de la primaria. **Es halo.**
- **(c) Por qué el rojo es más alto en valor absoluto:** el suelo sigue la SED de la
  primaria. La primaria sube **×8.7** de 4750→9350 Å (K + A_V≈1.8), el suelo sube solo
  **×2.35**, luego la *fracción* de halo que entra en el cielo **baja ×0.27** hacia el rojo —
  el halo AO cromático ([Fetick et al. 2019](https://doi.org/10.1051/0004-6361/201935830), el
  mismo modelo que usa C1/E1): a mayor λ, mejor Strehl, más luz en el core y menos en el
  halo. Los dos efectos se oponen y gana la SED → **suelo más alto en el rojo**.
- **(d) Por qué el azul *parece* limpio:** en unidades del ruido del propio canal, el suelo
  vale **0.31 σ** en el azul y **~1.1 σ** en el rojo. En el azul la respuesta de MUSE hunde
  la transmisión, el RMS por canal sube a ~3–4.6 y el suelo queda **enterrado**; en el rojo
  el RMS baja a ~1.6 y el mismo tipo de residuo **asoma al nivel del ruido**. No es que el
  cielo esté peor restado en el rojo: es que allí se *ve*.

**Consecuencia para la cadena** (no para ZAP, que sigue siendo innecesario): este suelo es
aditivo y crece hacia el rojo, así que hay que quitarlo antes de integrar flujo en aperturas
grandes — es exactamente lo que mide `scripts/measure_growth_curve.py` para `apcorr` y lo
que el fondo de anillo corrige en C2. Y explica que el azul del espectro final sea el tramo
menos citable: allí el residuo está dentro del ruido, no verificado.


In [ ]:
if MAKE_PLOT:
    try:
        import glob as _glob
        from pathlib import Path
        import numpy as np
        import matplotlib.pyplot as plt
        from astropy.io import fits
        from musepipe.reduction.sky_zap import channel_rms_from_mask
        from musepipe.reduction.verify import circular_aperture_mask

        A = _a2_load()
        wave, data, wl = A["wave"], A["data"], A["wl"]
        sky_mask, sky_med = A["sky_mask"], A["sky_med"]
        _sy, _sx = A["star_yx"]
        _root = Path(A["cube_path"]).parent.parent

        # 1) La evidencia literal de que el DRS resto cielo, del log de esorex.
        _logs = sorted(_glob.glob(str(_root / "*" / "products" / "*" / "muse_scipost.log")))
        print("=== que hizo muse_scipost con el cielo (log de esorex, literal) ===")
        if _logs:
            print(f"log: {_logs[0]}")
            for _ln in Path(_logs[0]).read_text(errors="replace").splitlines():
                if any(k in _ln for k in ("sky mask", "Sky line fit", "sky continuum",
                                          "sky subtraction", "Cutting data", "sky-spectrum",
                                          "Loaded sky lines")):
                    print("   ", _ln.split("] ")[-1].strip())
        else:
            print("   no hay log de muse_scipost bajo", _root)
            print("   (cubo ADP o workdir de reduccion no disponible: no re-derivable aqui)")

        # 2) Lo que el DRS llamo cielo y lo que resto.
        _mk = sorted(_glob.glob(str(_root / "*" / "products" / "*" / "SKY_MASK_*.fits")))
        _ct = sorted(_glob.glob(str(_root / "*" / "products" / "*" / "SKY_CONTINUUM_*.fits")))
        _sp = sorted(_glob.glob(str(_root / "*" / "products" / "*" / "SKY_SPECTRUM_*.fits")))
        if _mk:
            _m = np.asarray(fits.getdata(_mk[0]))
            print(f"\nSKY_MASK del DRS: {(_m > 0).mean():.3f} del campo"
                  f"  (skymodel_fraction=0.1 tras ignorar 0.05)"
                  f" vs {sky_mask.mean():.3f} de la zona de cielo de A2")
        _wc = _fc = None
        if _ct:
            _t = fits.open(_ct[0])[1].data
            _wc, _fc = np.asarray(_t["lambda"], float), np.asarray(_t["flux"], float)
            print(f"SKY_CONTINUUM restado: {np.median(_fc[(_wc >= 4750) & (_wc < 5000)]):+.2f} (azul)"
                  f" -> {np.median(_fc[(_wc >= 8000) & (_wc < 8300)]):+.2f} (8000-8300 A)"
                  f"   frente a un suelo residual de {np.nanmedian(sky_med):+.2f}")

        # 3) Perfil radial del suelo: cielo mal restado = plano; halo = cae.
        yy, xx = np.indices(wl.shape)
        rr = np.hypot(yy - _sy, xx - _sx)
        _r0 = float(np.floor(rr[sky_mask].min()))
        ann = [(_r0 + k * 12, _r0 + (k + 1) * 12) for k in range(5)]
        bands = [(wave[0], wave[0] + 550), (6500, 7050), (wave[-1] - 550, wave[-1])]
        _prof = {}
        print("\n=== perfil radial del suelo residual (anillos fuera de la mascara) ===")
        for a, b in bands:
            sel = (wave >= a) & (wave <= b)
            img = np.nanmedian(data[sel], axis=0)
            vals = []
            for q0, q1 in ann:
                s = sky_mask & (rr >= q0) & (rr < q1)
                vals.append(float(np.nanmedian(img[s])) if s.sum() > 20 else np.nan)
            _prof[(a, b)] = vals
            _fin = [v for v in vals if np.isfinite(v)]
            _dr = _fin[0] / _fin[-1] if len(_fin) > 1 and _fin[-1] else np.nan
            print(f"  {a:6.0f}-{b:6.0f} A: " + " ".join(f"{v:+6.3f}" for v in vals)
                  + f"   caida dentro/fuera = {_dr:.2f}x")
        print("  -> cae con el radio en las tres bandas: el suelo es HALO de la primaria,"
              " no cielo mal restado.")

        # 4) Suelo vs SED de la primaria: la fraccion de halo es cromatica.
        star_m = circular_aperture_mask(wl.shape, (_sy, _sx), 10.0)
        _idx = np.where(star_m.ravel())[0]
        star = np.full(A["n3"], np.nan)
        for c0 in range(0, A["n3"], 200):
            blk = data[c0:c0 + 200].reshape(min(200, A["n3"] - c0), -1)[:, _idx]
            star[c0:c0 + blk.shape[0]] = np.nansum(blk, axis=1)
        # La banda del laser AO se excluye de TODA metrica de decision (spec_A2 2.5);
        # aqui tambien, o el cociente suelo/primaria explota en sus bordes.
        from musepipe.reduction.sky_zap import NALGS_RANGE
        _nal = (wave >= NALGS_RANGE[0]) & (wave <= NALGS_RANGE[1])
        _f = np.isfinite(sky_med) & np.isfinite(star) & (star > 0) & ~_nal
        _blue = _f & (wave < wave[0] + 550); _red = _f & (wave > wave[-1] - 650)
        _fs, _ff = np.median(star[_red]) / np.median(star[_blue]), np.median(sky_med[_red]) / np.median(sky_med[_blue])
        print(f"\nazul -> rojo:  primaria x{_fs:.2f}   suelo x{_ff:.2f}"
              f"   fraccion de halo x{_ff / _fs:.2f}  (Strehl AO sube con lambda: Fetick+2019)")

        # 5) El suelo en unidades del ruido del canal: por que el azul parece limpio.
        rms = channel_rms_from_mask(data.astype(np.float64), sky_mask)
        _snr = sky_med / rms
        print(f"suelo / RMS del canal:  azul {np.nanmedian(_snr[_blue]):.2f} sigma"
              f"   rojo {np.nanmedian(_snr[_red]):.2f} sigma"
              f"   (RMS azul {np.nanmedian(rms[_blue]):.2f} vs rojo {np.nanmedian(rms[_red]):.2f})")

        fig, axes = plt.subplots(2, 2, figsize=(14.5, 8.6))
        ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

        # (a) lo restado por el DRS vs lo que quedo
        if _wc is not None:
            ax1.plot(_wc, _fc, lw=0.6, color="tab:blue", label="SKY_CONTINUUM restado por el DRS")
        if _sp:
            _t2 = fits.open(_sp[0])[1].data
            ax1.plot(_t2["lambda"], _t2["data"], lw=0.3, color="0.6", alpha=0.7,
                     label="SKY_SPECTRUM (cielo medido por el DRS)")
        ax1.plot(wave, sky_med, lw=0.6, color="tab:purple", label="suelo que QUEDA (zona de cielo A2)")
        ax1.axhline(0, color="tab:red", ls="--", lw=0.9)
        ax1.set_yscale("symlog", linthresh=1.0)
        ax1.set_xlabel("lambda [A]"); ax1.set_ylabel("flujo (unidades del cubo)")
        ax1.set_title("(a) el DRS SI resto cielo (skymethod=model, 9.1% mas oscuro del campo);\n"
                      "lo que queda es dos ordenes menor y de otro origen", fontsize=9.5)
        ax1.legend(fontsize=7)
        if _wc is None and not _sp:
            ax1.text(0.5, 0.5, "productos SKY_* del DRS no disponibles\npara esta cadena",
                     transform=ax1.transAxes, ha="center", va="center", fontsize=9, color="0.4")

        # (b) la prueba: plano (cielo) o decreciente (halo)
        _rc = [0.5 * (q0 + q1) for q0, q1 in ann]
        for (a, b), vals in _prof.items():
            ax2.plot(_rc, vals, "o-", lw=1.2, ms=4, label=f"{a:.0f}-{b:.0f} A")
        ax2.axhline(0, color="0.6", ls=":", lw=0.9)
        ax2.set_xlabel("radio desde la primaria [px]")
        ax2.set_ylabel("mediana del suelo en el anillo")
        if A["pix"] > 0:
            _tw = ax2.secondary_xaxis("top", functions=(lambda v: v * A["pix"], lambda v: v / A["pix"]))
            _tw.set_xlabel("radio [arcsec]", fontsize=8); _tw.tick_params(labelsize=7)
        ax2.set_title("(b) el suelo CAE con el radio -> es halo de la primaria,\n"
                      "no cielo mal restado (un residuo de cielo seria plano)", fontsize=9.5)
        ax2.legend(fontsize=7.5)

        # (c) suelo vs SED de la primaria
        _n = lambda v, s: v / np.nanmedian(v[s])
        ax3.plot(wave[_f], _n(star, _blue)[_f], lw=0.5, color="tab:orange",
                 label=f"primaria (normalizada al azul) x{_fs:.1f}")
        ax3.plot(wave[_f], _n(sky_med, _blue)[_f], lw=0.5, color="tab:purple",
                 label=f"suelo (normalizado al azul) x{_ff:.1f}")
        ax3.set_yscale("log"); ax3.set_xlabel("lambda [A]"); ax3.set_ylabel("normalizado al azul")
        _axr = ax3.twinx()
        _rat = (sky_med / star)
        _rat = _rat / np.nanmedian(_rat[_blue])
        _axr.plot(wave[_f], _rat[_f], lw=0.5, color="tab:green")
        _axr.set_ylim(0, np.nanpercentile(_rat[_f], 99.5) * 1.1)
        _axr.set_ylabel("fraccion de halo en el cielo (rel. al azul)", color="tab:green", fontsize=8)
        _axr.tick_params(axis="y", colors="tab:green", labelsize=7)
        ax3.set_title("(c) el suelo sigue la SED de la primaria, pero mas plano:\n"
                      "la fraccion de halo BAJA hacia el rojo (Strehl AO cromatico)", fontsize=9.5)
        ax3.legend(fontsize=7.5, loc="upper left")

        # (d) el suelo en unidades de ruido
        ax4.plot(wave, _snr, lw=0.5, color="0.35")
        ax4.axhline(1.0, color="tab:red", ls="--", lw=1.0, label="1 sigma del canal")
        ax4.axhline(0.0, color="0.6", ls=":", lw=0.9)
        ax4.set_ylim(np.nanpercentile(_snr, 0.5) - 0.2, np.nanpercentile(_snr, 99.5) + 0.2)
        ax4.set_xlabel("lambda [A]"); ax4.set_ylabel("suelo / RMS del canal [sigma]")
        ax4.set_title("(d) por que el azul PARECE limpio: alli el suelo esta enterrado\n"
                      "en el ruido de la respuesta; en el rojo asoma a ~1 sigma", fontsize=9.5)
        ax4.legend(fontsize=7.5)

        fig.tight_layout(rect=[0, 0.055, 1, 1])
        fig.text(0.005, 0.012, A2_REFS + "  ·  halo AO cromatico: Fetick et al. 2019, A&A 628, A99",
                 fontsize=6.2, color="0.25", wrap=True)
        outdir = nb.run_dir(RUN_ID) / "plots" / "a2_m4"
        outdir.mkdir(parents=True, exist_ok=True)
        fig.savefig(outdir / "cielo_drs_azul_rojo.png", dpi=110)
        print("\nfigura ->", outdir / "cielo_drs_azul_rojo.png")
        plt.show()
    except NameError:
        print("Ejecuta primero la celda del plot M4 (define _a2_load y A2_REFS).")
    except Exception as e:
        print("No se pudo generar el plot:", type(e).__name__, e)


## Decisiones y notas
- La métrica M4 (`R`) y la caracterización del airglow viven en A4/`stage00q_qc.json`; la regla de decisión, en `musepipe/reduction/sky_zap.py`.


## Conclusión (registrada)

**Decisión de este objeto: `zap_applied = False` (`not_needed`), con R = 0.6065899752113613 sobre la máscara de A2.** `n/d` = A2 no ha corrido para esta cadena: la decisión de abajo aún no está registrada para el objeto.

- **Datos:** cubo NFM-AO auto-reducido `cube_telcorr.fits` + `SKY_SPECTRUM` cacheado para caracterizar el airglow (el detalle del OB y las exposiciones de **este** objeto sale del QC de A1/A4; la celda de setup imprime de qué run vienen).
- **Evidencia:** `R = 0.5715509857697371` frente al umbral `1.5` (estado M4 = **yellow**); el cubo restado de cielo tiene pocas skylines usables (residual al nivel de ruido). En el campo diminuto NFM, ZAP aportaría ~0 y arriesgaría absorber señal del compañero.
- **Salvedad de M4:** el residuo es bajo, pero la escasez de skylines hace la métrica menos robusta que en WFM. No bloqueante.
- **Para el paper:** registrar como decisión con su métrica (R), **no** como omisión. Nada es paper-válido hasta cerrar el A-block del objeto.
